## 1. Generating the Synthetic Training Dataset

In [19]:

import pandas as pd
import random as rd
import numpy as np

rd.seed(42)

ranges = {
    "Highly Ready": {"bucket_A": (80,100), "bucket_B": (0,3)},
    "Moderately Ready": {"bucket_A": (70,79), "bucket_B": (4,5)},
    "Needs Improvement": {"bucket_A": (51,69), "bucket_B": (6,8)},
    "Not Ready Yet": {"bucket_A": (0,50), "bucket_B": (9,10)}
}

bucket_a_cols = ['skill_match_percentage', 'critical_skill_match_percentage','project_relevance_score', 
                 'certification_relevance_score', 'internship_relevance_score', 'resume_completeness_score', 
                 'keyword_match_score', 'role_category_match_score']

bucket_b_cols = ['missing_skills_count','critical_missing_skills_count']

all_rows = []

for k in range(2000):
    row = {}
    x = rd.choice(list(ranges.keys()))

    bucket_a_range = ranges[x]['bucket_A']
    low = bucket_a_range[0]  
    high = bucket_a_range[1]

    bucket_b_range = ranges[x]['bucket_B']
    low_b = bucket_b_range[0]
    high_b = bucket_b_range[1]

    # Generate a random float between 0.0 and 1.0
    chance = rd.random()

    if chance < 0.9:
        # 90% chance: Keep the original label 'x'
        label_to_use = x
    else:
        # 10% chance: Choose a label OTHER than 'x'
        other_labels = [label for label in ranges.keys() if label != x]
        label_to_use = rd.choice(other_labels)

    for i in bucket_a_cols:
        row[i] = rd.uniform(low, high)

    for i in bucket_b_cols:
        row[i] = rd.randint(low_b, high_b)

    row["readiness_label"] = label_to_use
    all_rows.append(row)

df = pd.DataFrame(all_rows)

df.to_csv('student_data.csv', index=False)


## 2. Verifying the Generated Dataset

In [20]:
df = pd.read_csv('student_data.csv')
print(df.shape)
print(df['readiness_label'].value_counts())

(2000, 11)
readiness_label
Highly Ready         512
Not Ready Yet        501
Needs Improvement    495
Moderately Ready     492
Name: count, dtype: int64


## 3. Loading Data and Creating Train/Test Split

In [21]:
df = pd.read_csv(r'D:\G_PRIS\student_data.csv')

X = df[['skill_match_percentage', 'critical_skill_match_percentage',
     'project_relevance_score', 'certification_relevance_score', 
     'internship_relevance_score', 'resume_completeness_score', 
     'keyword_match_score', 'role_category_match_score', 
     'missing_skills_count','critical_missing_skills_count']]

y = df['readiness_label']

from sklearn.model_selection import train_test_split 
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 4. Training the Logistic Regression ModelTraining

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

                   precision    recall  f1-score   support

     Highly Ready       0.90      0.86      0.88        95
 Moderately Ready       0.76      0.93      0.83       100
Needs Improvement       0.91      0.77      0.83       111
    Not Ready Yet       0.87      0.84      0.85        94

         accuracy                           0.85       400
        macro avg       0.86      0.85      0.85       400
     weighted avg       0.86      0.85      0.85       400



## 5. Saving the Trained Model

In [23]:
import joblib
joblib.dump(model, 'logistic_model.joblib')

['logistic_model.joblib']

## 6. Installing Required Packages for the LLM Pipeline

In [24]:
%pip install python-dotenv
%pip install pypdf
%pip install groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 7. Setting Up the Groq API Client

In [25]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=groq_api_key)

## 8. Extracting Text from a Sample Resume PDF

In [26]:
import pypdf

reader = pypdf.PdfReader("g_resume.pdf")
text = ""

for page in reader.pages:
    text += page.extract_text()

## 9. Sample Job Description for Testing

In [27]:
jd_text = '''This role delivers engaging online training sessions while providing 
            empathetic customer support to resolve daily technical and program inquiries. 
            You will guide learners through product onboarding, troubleshoot system issues 
            via chat and email, and create clear instructional materials to ensure student success. 
            The ideal candidate blends professional teaching or tutoring experience with a patient, 
            customer-first approach to problem-solving.'''

## 10. Defining Feature Order for Model Input

In [28]:
feature_order = ['skill_match_percentage', 'critical_skill_match_percentage',
     'project_relevance_score', 'certification_relevance_score', 
     'internship_relevance_score', 'resume_completeness_score', 
     'keyword_match_score', 'role_category_match_score', 
     'missing_skills_count','critical_missing_skills_count']

## 11. Final Analysis Function (Resume + JD → Prediction)

In [29]:
import json

def analyze_resume(resume_text, jd_text):
    # Step 1: build the prompt (reuse your JSON template, just swap {text} for resume_text)
        prompt = f'''act as an expert resume scanner to read {resume_text} and {jd_text} and 
                    give score,percentage and count for the numerical features 
                    {{
                        "skill_match_percentage": 75,
                        "critical_skill_match_percentage": 75,
                        "project_relevance_score": 75,
                        "certification_relevance_score": 75,
                        "internship_relevance_score": 75,
                        "resume_completeness_score": 75,
                        "keyword_match_score": 75,
                        "role_category_match_score": 75,
                        "missing_skills_count": 6,
                        "critical_missing_skills_count": 8,
                        "missing_skills": ["Docker", "SQL"],
                        "critical_missing_skills": ["Docker", "SQL"],
                        "matched_skills": ["Docker", "SQL"],
                        "feedback": "Your resume shows strong technical skills but lacks SQL",
                        "roadmap_7_day": "Complete SQL course and build 2 projects using it",
                        "roadmap_30_day": "Complete SQL course and build 2 projects using it",
                        "Resume_improvement_suggestion": "Complete SQL course and build 2 projects using it",
                        "Job-specific_preparation_suggestions": "Complete SQL course and build 2 projects using it"
                    }}
                    Analyze the actual resume and job description below, then return ONLY a JSON 
                    object in the following format, with your own real values replacing the examples'''
        
        # Step 2: call Groq
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content":prompt}],
            response_format={"type": "json_object"}
        )
        
        # Step 3: parse JSON
        data = json.loads(response.choices[0].message.content)
        print(data)

        # Step 4: fix count mismatches
        data["missing_skills"] = data.get("missing_skills", [])
        data["critical_missing_skills"] = data.get("critical_missing_skills", [])
        data["matched_skills"] = data.get("matched_skills", [])
        data["missing_skills_count"] = len(data["missing_skills"])
        data["critical_missing_skills_count"] = len(data["critical_missing_skills"])

        readiness_score = sum(data[col] for col in bucket_a_cols) / len(bucket_a_cols)
        data["placement_readiness_score"] = round(readiness_score, 2)
        
        # Step 5: build input row for the model
        input_row = pd.DataFrame([data])[feature_order]
        
        # Step 6: predict
        prediction = model.predict(input_row.values)
                
        
        # Step 7: add prediction into data
        data["predicted_readiness_label"] = prediction[0]
        
        # Step 8: return everything
        return data

result = analyze_resume(text, jd_text)
print(json.dumps(result, indent=2))


{'skill_match_percentage': 60, 'critical_skill_match_percentage': 50, 'project_relevance_score': 40, 'certification_relevance_score': 30, 'internship_relevance_score': 20, 'resume_completeness_score': 80, 'keyword_match_score': 70, 'role_category_match_score': 60, 'missing_skills_count': 10, 'critical_missing_skills_count': 12, 'missing_skills': ['Technical Support', 'Customer Service', ' Troubleshooting', 'Streaming Device', 'Hardware', 'Software', 'Network Issues', 'SQL', 'Data Analysis', 'Programming'], 'critical_missing_skills': ['Technical Support', 'Customer Service', 'Troubleshooting', 'Streaming Device', 'Hardware', 'Software', 'Network Issues', 'SQL', 'Data Analysis', 'Programming', 'Cloud Computing', 'IOT'], 'matched_skills': ['Teaching', 'Lesson Planning', 'Communication', 'Problem-Solving', 'Leadership'], 'feedback': 'Your resume shows strong teaching skills but lacks technical support and customer service experience', 'roadmap_7_day': 'Complete a technical support course a